In [1]:
# Storm Stats
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from rasterstats import zonal_stats


shapefile = "../../public/shapefiles/county.shp"

In [7]:
import rasterio
import numpy as np

for date in np.arange(17, 23+1):
    print(f"Processing date: 2026-03-{date:02d} cumulative")
    tif_path = f"./storm_data/2026_03_{date:02d}_cumulative.tif"

    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(float)

        # Handle nodata
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

        # Compute stats (convert mm → inches here too)
        min_val = np.nanmin(arr) / 25.4
        mean_val = np.nanmean(arr) / 25.4
        max_val = np.nanmax(arr) / 25.4

        print({
            "min": round(min_val, 1),
            "mean": round(mean_val, 1),
            "max": round(max_val, 1)
        })

Processing date: 2026-03-17 cumulative
{'min': np.float64(0.0), 'mean': np.float64(0.8), 'max': np.float64(5.7)}
Processing date: 2026-03-18 cumulative
{'min': np.float64(0.0), 'mean': np.float64(1.0), 'max': np.float64(5.8)}
Processing date: 2026-03-19 cumulative
{'min': np.float64(0.1), 'mean': np.float64(1.5), 'max': np.float64(9.1)}
Processing date: 2026-03-20 cumulative
{'min': np.float64(0.1), 'mean': np.float64(2.4), 'max': np.float64(22.1)}
Processing date: 2026-03-21 cumulative
{'min': np.float64(0.2), 'mean': np.float64(3.3), 'max': np.float64(23.7)}
Processing date: 2026-03-22 cumulative
{'min': np.float64(0.3), 'mean': np.float64(3.9), 'max': np.float64(23.8)}
Processing date: 2026-03-23 cumulative
{'min': np.float64(0.3), 'mean': np.float64(4.6), 'max': np.float64(28.0)}


In [10]:
import rasterio
import numpy as np

for date in np.arange(17, 23+1):
    print(f"Processing date: 2026-03-{date:02d} daily")
    tif_path = f"./storm_data/2026_03_{date:02d}.tif"

    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(float)

        # Handle nodata
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

        # Compute stats (convert mm → inches here too)
        min_val = np.nanmin(arr) / 25.4
        mean_val = np.nanmean(arr) / 25.4
        max_val = np.nanmax(arr) / 25.4

        print({
            "min": round(min_val, 1),
            "mean": round(mean_val, 1),
            "max": round(max_val, 1)
        })

Processing date: 2026-03-17 daily
{'min': np.float64(0.0), 'mean': np.float64(0.8), 'max': np.float64(5.7)}
Processing date: 2026-03-18 daily
{'min': np.float64(0.0), 'mean': np.float64(0.1), 'max': np.float64(1.0)}
Processing date: 2026-03-19 daily
{'min': np.float64(0.0), 'mean': np.float64(0.5), 'max': np.float64(8.9)}
Processing date: 2026-03-20 daily
{'min': np.float64(0.0), 'mean': np.float64(1.0), 'max': np.float64(13.1)}
Processing date: 2026-03-21 daily
{'min': np.float64(0.0), 'mean': np.float64(0.8), 'max': np.float64(9.6)}
Processing date: 2026-03-22 daily
{'min': np.float64(0.0), 'mean': np.float64(0.6), 'max': np.float64(8.9)}
Processing date: 2026-03-23 daily
{'min': np.float64(0.0), 'mean': np.float64(0.7), 'max': np.float64(7.3)}


In [12]:
import geopandas as gpd
import numpy as np
import pandas as pd
from rasterstats import zonal_stats

gdf = gpd.read_file(shapefile).dissolve(by="county", as_index=False).reset_index(drop=True)

records = []


for date in np.arange(17, 23 + 1):
  print(f"Processing date: 2026-03-{date:02d}")
  tif_path = f"./storm_data/2026_03_{date:02d}.tif"

  for county in gdf["county"].unique():
      county_gdf = gdf[gdf["county"] == county]

      stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

      # zonal_stats returns a list (one feature), so grab first dict
      s = stats[0]

      record = {
          "date": f"2026-03-{date:02d}",
          "county": county,
          "type": "daily",
          "min_in": round(s["min"] / 25.4, 1) if s["min"] is not None else None,
          "mean_in": round(s["mean"] / 25.4, 1) if s["mean"] is not None else None,
          "max_in": round(s["max"] / 25.4, 1) if s["max"] is not None else None,

      }

      records.append(record)
      


for date in np.arange(17, 23 + 1):
    print(f"Processing date: 2026-03-{date:02d} cumulative")
    tif_path = f"./storm_data/2026_03_{date:02d}_cumulative.tif"

    for county in gdf["county"].unique():
        county_gdf = gdf[gdf["county"] == county]

        stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

        # zonal_stats returns a list (one feature), so grab first dict
        s = stats[0]

        record = {
            "date": f"2026-03-{date:02d}",
            "county": county,
            "type": "cumulative",
            "min_in": round(s["min"] / 25.4, 1) if s["min"] is not None else None,
            "mean_in": round(s["mean"] / 25.4, 1) if s["mean"] is not None else None,
            "max_in": round(s["max"] / 25.4, 1) if s["max"] is not None else None,

        }

        records.append(record)


# Convert to DataFrame
df = pd.DataFrame(records)

print(df)

Processing date: 2026-03-17
Processing date: 2026-03-18
Processing date: 2026-03-19
Processing date: 2026-03-20
Processing date: 2026-03-21
Processing date: 2026-03-22
Processing date: 2026-03-23
Processing date: 2026-03-17 cumulative
Processing date: 2026-03-18 cumulative
Processing date: 2026-03-19 cumulative
Processing date: 2026-03-20 cumulative
Processing date: 2026-03-21 cumulative
Processing date: 2026-03-22 cumulative
Processing date: 2026-03-23 cumulative
          date    county        type  min_in  mean_in  max_in
0   2026-03-17   Hawaiʻi       daily     0.1      1.1     5.7
1   2026-03-17  Honolulu       daily     0.0      0.2     0.7
2   2026-03-17    Kauaʻi       daily     0.0      0.1     0.3
3   2026-03-17      Maui       daily     0.1      0.7     2.8
4   2026-03-18   Hawaiʻi       daily     0.0      0.2     1.0
5   2026-03-18  Honolulu       daily     0.0      0.0     0.7
6   2026-03-18    Kauaʻi       daily     0.0      0.0     0.1
7   2026-03-18      Maui       dail

In [17]:
tif_path = f"./storm_data/all_days_cumu_rf.tif"
records = []
gdf = gpd.read_file(shapefile).dissolve(by="county", as_index=False).reset_index(drop=True)
for county in gdf["county"].unique():
  county_gdf = gdf[gdf["county"] == county]

  stats = zonal_stats(county_gdf, tif_path, stats=["mean", "min", "max"])

  s = stats[0]

  record = {
      "county": county,
      "min_in": round(s["min"] / 25.4, 1) if s["min"] is not None else None,
      "mean_in": round(s["mean"] / 25.4, 1) if s["mean"] is not None else None,
      "max_in": round(s["max"] / 25.4, 1) if s["max"] is not None else None,
  }

  records.append(record)

records

[{'county': 'Hawaiʻi', 'min_in': 5.6, 'mean_in': 15.1, 'max_in': 34.8},
 {'county': 'Honolulu', 'min_in': 13.4, 'mean_in': 24.2, 'max_in': 54.5},
 {'county': 'Kauaʻi', 'min_in': 2.2, 'mean_in': 12.1, 'max_in': 26.2},
 {'county': 'Maui', 'min_in': 12.1, 'mean_in': 25.4, 'max_in': 76.0}]

In [18]:
tif_path = f"./storm_data/all_days_cumu_rf.tif"

with rasterio.open(tif_path) as src:
    arr = src.read(1).astype(float)

    # Handle nodata
    if src.nodata is not None:
        arr = np.where(arr == src.nodata, np.nan, arr)

    # Compute stats (convert mm → inches here too)
    min_val = np.nanmin(arr) / 25.4
    mean_val = np.nanmean(arr) / 25.4
    max_val = np.nanmax(arr) / 25.4

    print({
        "min": round(min_val, 1),
        "mean": round(mean_val, 1),
        "max": round(max_val, 1)
    })

{'min': np.float64(2.2), 'mean': np.float64(17.6), 'max': np.float64(76.0)}
